# 03 — Live Monitoring
Runs the current-status pipeline for both cities and prints a formatted dashboard.

In [ ]:
import sys; sys.path.insert(0,'..')
import warnings; warnings.filterwarnings('ignore')
from src.data_ingestion import ingest_city_data
from src.preprocessing import clean, aggregate_time_windows, compute_rolling_baseline, city_level_aggregate
from src.signal_extraction import extract_latest_signals, extract_all_signals
from src.decision_engine import evaluate_all_signals
print('Ready')

In [ ]:
for city in ['nyc','boston']:
    raw=ingest_city_data(city,days_back=45)
    cl=clean(raw); agg=aggregate_time_windows(cl,freq='h')
    base=compute_rolling_baseline(agg,freq='h'); city_agg=city_level_aggregate(base,freq='h')
    signals=extract_latest_signals(city_agg,base)
    print(f'\n{city.upper()} — Current ({raw["data_source"].iloc[0]})')
    for s in sorted(signals,key=lambda x:-x.activity_ratio):
        print(f'  {s.category:<18} {s.activity_ratio:>5.2f}x  z={s.z_score:>+5.1f}  {s.trend}')

In [ ]:
# Decision run
for city in ['nyc','boston']:
    raw=ingest_city_data(city,days_back=45)
    cl=clean(raw); agg=aggregate_time_windows(cl,freq='h')
    base=compute_rolling_baseline(agg,freq='h'); city_agg=city_level_aggregate(base,freq='h')
    signals=extract_all_signals(city_agg,base,stride=12)
    results,engine=evaluate_all_signals(signals)
    c=engine.memory.action_counts()
    print(f'\n{city.upper()}: ACT={c.get("ACT",0)} DEFER={c.get("DEFER",0)} WAIT={c.get("WAIT",0)}')
    for s,d in results:
        if d.action=='ACT': print(f'  → {s.category} | {s.activity_ratio:.2f}x | {d.confidence}')